In [1]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv(r"..\data\raw\house_price.csv")

def safe_div(numerator, denominator):
    return (
        numerator / denominator.replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0)

In [ ]:
# Mede a idade da casa no momento da venda.
# Casas mais novas tendem a ter preços maiores, e idade é mais interpretável que o ano bruto.
df["HouseAge"] = (df["YrSold"] - df["YearBuilt"]).clip(lower=0)

# Mede há quantos anos a casa foi reformada no momento da venda.
# Captura se a casa estava mais atualizada ou antiga quando foi vendida.
df["RemodAge"] = (df["YrSold"] - df["YearRemodAdd"]).clip(lower=0)

# Indica se a casa passou por reforma desde sua construção.
# Se YearRemodAdd for diferente de YearBuilt, houve alguma atualização relevante.
df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)

# Indica se a casa foi vendida no mesmo ano em que foi construída.
# Pode capturar imóveis novos, que geralmente têm comportamento diferente no preço.
df["IsNewHouse"] = (df["YrSold"] == df["YearBuilt"]).astype(int)

# Mede a idade da garagem no momento da venda.
# Quando não existe garagem, GarageYrBlt costuma ser 0 ou nulo, então a idade é definida como 0.
garage_year = df["GarageYrBlt"].fillna(0)
df["GarageAge"] = np.where(
    garage_year > 0,
    (df["YrSold"] - garage_year).clip(lower=0),
    0
)

# Soma toda a área construída principal da casa, incluindo porão e área acima do solo.
# Captura o tamanho total utilizável/construído de forma mais completa que GrLivArea isoladamente.
df["TotalSF"] = (
    df["TotalBsmtSF"].fillna(0) +
    df["1stFlrSF"].fillna(0) +
    df["2ndFlrSF"].fillna(0) +
    df["LowQualFinSF"].fillna(0)
)

# Soma apenas as áreas finalizadas do porão.
# Diferencia casas com porão acabado de casas com porão grande, mas pouco aproveitado.
df["TotalFinishedBsmtSF"] = (
    df["BsmtFinSF1"].fillna(0) +
    df["BsmtFinSF2"].fillna(0)
)

# Mede a proporção do porão que está finalizada.
# Ajuda o modelo a entender qualidade de aproveitamento do porão, não apenas tamanho bruto.
df["BsmtFinishedRatio"] = safe_div(
    df["TotalFinishedBsmtSF"],
    df["TotalBsmtSF"].fillna(0)
)

# Soma todos os banheiros, dando peso 0.5 para lavabos.
# Representa melhor o conforto total da casa do que olhar cada tipo de banheiro separadamente.
df["TotalBath"] = (
    df["FullBath"].fillna(0) +
    0.5 * df["HalfBath"].fillna(0) +
    df["BsmtFullBath"].fillna(0) +
    0.5 * df["BsmtHalfBath"].fillna(0)
)

# Soma todas as áreas de varanda.
# Reduz várias colunas esparsas em uma medida única de área de varanda.
df["TotalPorchSF"] = (
    df["OpenPorchSF"].fillna(0) +
    df["EnclosedPorch"].fillna(0) +
    df["3SsnPorch"].fillna(0) +
    df["ScreenPorch"].fillna(0)
)

# Soma áreas externas valorizáveis: deck, varandas e piscina.
# Captura o espaço externo total aproveitável da propriedade.
df["TotalOutdoorSF"] = (
    df["WoodDeckSF"].fillna(0) +
    df["OpenPorchSF"].fillna(0) +
    df["EnclosedPorch"].fillna(0) +
    df["3SsnPorch"].fillna(0) +
    df["ScreenPorch"].fillna(0) +
    df["PoolArea"].fillna(0)
)

# Mede a área média da garagem por carro.
# Diferencia garagens apertadas de garagens mais espaçosas para a mesma capacidade.
df["GarageAreaPerCar"] = safe_div(
    df["GarageArea"].fillna(0),
    df["GarageCars"].fillna(0)
)

# Mede a área habitável média por cômodo acima do solo.
# Ajuda a separar casas com muitos cômodos pequenos de casas com cômodos maiores.
df["SFPerRoom"] = safe_div(
    df["GrLivArea"].fillna(0),
    df["TotRmsAbvGrd"].fillna(0)
)

# Mede a proporção de quartos em relação ao total de cômodos.
# Casas com muitos quartos e poucos espaços sociais podem ter dinâmica diferente de preço.
df["BedroomsPerRoom"] = safe_div(
    df["BedroomAbvGr"].fillna(0),
    df["TotRmsAbvGrd"].fillna(0)
)

# Mede a quantidade de banheiros por quarto.
# Captura conforto relativo, especialmente em casas maiores.
df["BathPerBedroom"] = safe_div(
    df["TotalBath"],
    df["BedroomAbvGr"].fillna(0) + 1
)

# Mede a proporção entre frente do terreno e área total.
# Ajuda a capturar formato/aproveitamento do lote, além de LotArea e LotFrontage isolados.
df["LotFrontageRatio"] = safe_div(
    df["LotFrontage"].fillna(0),
    df["LotArea"].fillna(0)
)

# Combina qualidade geral e condição geral da casa.
# OverallQual mede qualidade; OverallCond mede estado de conservação. A combinação captura o efeito conjunto.
df["QualCondScore"] = (
    df["OverallQual"].fillna(0) *
    df["OverallCond"].fillna(0)
)

# Combina qualidade geral com área habitável.
# Uma casa grande e de alta qualidade tende a valer mais que uma casa grande de baixa qualidade.
df["QualityLivingArea"] = (
    df["OverallQual"].fillna(0) *
    df["GrLivArea"].fillna(0)
)

# Combina qualidade geral com área total construída.
# Captura o efeito conjunto entre tamanho total e qualidade percebida da casa.
df["QualityTotalSF"] = (
    df["OverallQual"].fillna(0) *
    df["TotalSF"].fillna(0)
)

# Indica se a casa possui garagem.
# Útil porque várias casas têm valores 0 nas colunas de garagem.
df["HasGarage"] = (df["GarageArea"].fillna(0) > 0).astype(int)

# Indica se a casa possui lareira.
# A ausência de lareira pode ter significado próprio, diferente de apenas quantidade 0.
df["HasFireplace"] = (df["Fireplaces"].fillna(0) > 0).astype(int)

# Indica se a casa possui piscina.
# PoolArea é muito esparsa; a presença da piscina pode ser mais informativa que a área bruta.
df["HasPool"] = (df["PoolArea"].fillna(0) > 0).astype(int)

# Indica se a casa possui porão.
# Casas sem porão têm comportamento diferente de casas com porão pequeno.
df["HasBsmt"] = (df["TotalBsmtSF"].fillna(0) > 0).astype(int)

# Indica se a casa possui alguma área de varanda.
# Resume várias colunas de varanda com muitos zeros.
df["HasPorch"] = (df["TotalPorchSF"].fillna(0) > 0).astype(int)

# Indica se a casa possui deck de madeira.
# WoodDeckSF costuma ter muitos zeros; a presença do deck pode ser relevante.
df["HasDeck"] = (df["WoodDeckSF"].fillna(0) > 0).astype(int)

# Indica se a casa possui segundo andar.
# Ajuda a capturar diferenças de estrutura entre casas térreas e casas com mais pavimentos.
df["Has2ndFloor"] = (df["2ndFlrSF"].fillna(0) > 0).astype(int)

# Indica se a casa possui área de baixa qualidade finalizada.
# Mesmo pequena, essa área pode sinalizar pior aproveitamento da construção.
df["HasLowQualFinSF"] = (df["LowQualFinSF"].fillna(0) > 0).astype(int)

# Indica se a casa possui revestimento de alvenaria.
# MasVnrArea tem muitos zeros; a presença do revestimento pode ser mais informativa.
df["HasMasVnr"] = (df["MasVnrArea"].fillna(0) > 0).astype(int)

# Indica se existe cerca.
# Considera que valores nulos ou "None" representam ausência de cerca.
df["HasFence"] = (
    df["Fence"]
    .fillna("None")
    .ne("None")
    .astype(int)
)

# Indica se existe viela/acesso lateral.
# Considera que valores nulos ou "None" representam ausência de acesso por viela.
df["HasAlley"] = (
    df["Alley"]
    .fillna("None")
    .ne("None")
    .astype(int)
)

# Indica se existe alguma característica adicional.
# MiscFeature é esparsa; a presença da característica pode ser mais útil que várias categorias raras.
df["HasMiscFeature"] = (
    df["MiscFeature"]
    .fillna("None")
    .ne("None")
    .astype(int)
)

In [7]:
cols_to_drop = [
    "Id",
    "YearBuilt",
    "YearRemodAdd",
    "GarageYrBlt",
    "1stFlrSF",
    "2ndFlrSF",
    "LowQualFinSF",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "FullBath",
    "HalfBath",
    "BsmtFullBath",
    "BsmtHalfBath",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "WoodDeckSF",
    "PoolArea",
    "LotFrontage"
]

df = df.drop(columns=cols_to_drop)

In [8]:
df.to_csv(r"..\data\processed\V1\Feature_engineering.csv")